# Ford Mustang Mach-E 2026 — 360° Video Transcript → Pinecone
### Ingestion pipeline for voice/speech transcripts from training videos

**Sources:** `Talk/*_segments.json` — transcribed 360° walkthrough videos  
**Namespace:** `video_transcript_v2` (same Pinecone index: `ford-mache-erg`)  
**Chunking:** Sliding window of 10 segments, step 8 (2-segment overlap)  
**Metadata config:** `video_metadata.json` at project root

Re-run this notebook for any new `*_segments.json` files placed in `Talk/`  
(add a matching entry to `video_metadata.json` first).

## 1. Install Dependencies

In [ ]:
!pip install -q openai python-dotenv pinecone-client

## 2. Imports & Setup

In [ ]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY, PINECONE_API_KEY from .env

import openai
from pinecone import Pinecone, ServerlessSpec

print("Imports loaded.")

## 3. Configuration

In [ ]:
# ── Pinecone ───────────────────────────────────────────────────────────────
PINECONE_INDEX_NAME = "ford-mache-erg"
PINECONE_CLOUD      = "aws"
PINECONE_REGION     = "us-east-1"
NAMESPACE           = "video_transcript_v2"

# ── Source files ──────────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.getcwd())
TALK_DIR     = PROJECT_ROOT / "Talk"
META_PATH    = PROJECT_ROOT / "video_metadata.json"

# ── Chunking parameters ───────────────────────────────────────────────────
WINDOW_SIZE  = 10   # consecutive segments per chunk
STEP_SIZE    = 8    # advance by 8 → 2-segment overlap between consecutive chunks

# ── Embedding ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"
EMBED_BATCH     = 50

# ── Clients ───────────────────────────────────────────────────────────────
openai_client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
pc            = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

print(f"Talk dir      : {TALK_DIR}")
print(f"Metadata file : {META_PATH}")
print(f"Pinecone index: {PINECONE_INDEX_NAME}")
print(f"Namespace     : {NAMESPACE}")
print(f"Window/step   : {WINDOW_SIZE}/{STEP_SIZE} ({WINDOW_SIZE - STEP_SIZE} seg overlap)")
print("Configuration loaded.")

## 4. Load Video Metadata Config

In [ ]:
with open(META_PATH, "r", encoding="utf-8") as f:
    VIDEO_META = json.load(f)

print(f"Loaded metadata for {len(VIDEO_META)} video(s):")
for stem, m in VIDEO_META.items():
    print(f"  {stem}")
    print(f"    video_id      : {m['video_id']}")
    print(f"    title         : {m['title']}")
    print(f"    duration_hms  : {m['duration_hms']}")
    print(f"    whisper_model : {m['whisper_model']}")
    print(f"    transcribed_at: {m['transcribed_at']}")

## 5. Pinecone Index Setup

In [ ]:
existing_indexes = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"Creating index '{PINECONE_INDEX_NAME}'...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
else:
    print(f"Using existing index '{PINECONE_INDEX_NAME}'")

index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())

## 6. Chunking Helper

Groups consecutive speech segments into overlapping windows.  
Field names match the enriched metadata schema (`chunk_start_hms`, `chunk_end_hms`, etc.).

In [ ]:
def sliding_window_chunks(segments, window=10, step=8):
    """
    Yield dicts, each representing one chunk:
        text                : concatenated speech text for this window
        chunk_start_seconds : float seconds at which this window begins
        chunk_end_seconds   : float seconds at which this window ends
        chunk_start_hms     : HH:MM:SS string
        chunk_end_hms       : HH:MM:SS string
        segment_ids         : "first_id-last_id" inclusive range
        chunk_index         : sequential 0-based index
    """
    n = len(segments)
    chunk_idx = 0
    for start in range(0, n, step):
        window_segs = segments[start : start + window]
        if not window_segs:
            break
        text = " ".join(s["text"].strip() for s in window_segs if s["text"].strip())
        if not text:
            chunk_idx += 1
            continue
        yield {
            "text":                text,
            "chunk_start_seconds": float(window_segs[0]["start"]),
            "chunk_end_seconds":   float(window_segs[-1]["end"]),
            "chunk_start_hms":     window_segs[0]["start_hms"],
            "chunk_end_hms":       window_segs[-1]["end_hms"],
            "segment_ids":         f"{window_segs[0]['id']}-{window_segs[-1]['id']}",
            "chunk_index":         chunk_idx,
        }
        chunk_idx += 1

print("Chunking helper defined.")

## 7. Embedding Helper

In [ ]:
def embed_texts(texts):
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]

print("Embedding helper defined.")

## 8. Full Ingestion Loop

Processes every `*_segments.json` file in `Talk/` that has a matching entry in `video_metadata.json`.  
Safe to re-run — Pinecone upsert is idempotent (same vector IDs overwrite).

In [ ]:
import glob as _glob

transcript_files = sorted(_glob.glob(str(TALK_DIR / "*_segments.json")))
if not transcript_files:
    print(f"No *_segments.json files found in {TALK_DIR}")
else:
    print(f"Found {len(transcript_files)} transcript file(s):")
    for f in transcript_files:
        stem = Path(f).stem.replace("_segments", "")
        has_meta = stem in VIDEO_META
        print(f"  {Path(f).name}  {'✅ metadata found' if has_meta else '⚠️  NO entry in video_metadata.json — will skip'}")

In [ ]:
total_upserted = 0
UPSERT_BATCH   = 100

for transcript_path in transcript_files:
    filename = Path(transcript_path).name
    stem     = Path(transcript_path).stem.replace("_segments", "")

    if stem not in VIDEO_META:
        print(f"\n⚠️  Skipping '{filename}' — no entry in video_metadata.json")
        continue

    meta = VIDEO_META[stem]
    print(f"\n{'─'*60}")
    print(f"📝  {filename}  →  namespace: {NAMESPACE}")
    print(f"    video_id: {meta['video_id']}  |  title: {meta['title']}")

    with open(transcript_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Support both old format {"segments":[...]} and new envelope
    segments = data["segments"] if isinstance(data, dict) else data
    print(f"  Loaded {len(segments)} segments")

    chunks = list(sliding_window_chunks(segments, window=WINDOW_SIZE, step=STEP_SIZE))
    print(f"  Created {len(chunks)} chunks (window={WINDOW_SIZE}, step={STEP_SIZE})")

    # ── Embed in batches ──────────────────────────────────────────────────
    vectors = []
    for batch_start in range(0, len(chunks), EMBED_BATCH):
        batch      = chunks[batch_start : batch_start + EMBED_BATCH]
        embeddings = embed_texts([c["text"] for c in batch])

        for chunk, embedding in zip(batch, embeddings):
            vector_id = f"{stem}_chunk{chunk['chunk_index']:04d}"
            vectors.append({
                "id":     vector_id,
                "values": embedding,
                "metadata": {
                    # ── identity ──────────────────────────────────────
                    "video_id":            meta["video_id"],
                    "source_file":         meta["source_file"],
                    "video_label":         meta["video_label"],
                    "title":               meta["title"],
                    "channel":             meta["channel"],
                    "video_url":           meta["video_url"]       or "",
                    "thumbnail_url":       meta["thumbnail_url"]   or "",
                    # ── temporal ──────────────────────────────────────
                    "upload_date_iso":     meta["upload_date_iso"],
                    "duration_hms":        meta["duration_hms"],
                    "duration_seconds":    meta["duration_seconds"],
                    # ── taxonomy ──────────────────────────────────────
                    "tags":                meta["tags"],
                    "categories":          meta["categories"],
                    "chapter":             meta.get("chapter")     or "",
                    # ── provenance ────────────────────────────────────
                    "whisper_model":       meta["whisper_model"],
                    "transcribed_at":      meta["transcribed_at"],
                    "view_count":          meta["view_count"]      or 0,
                    # ── chunk ─────────────────────────────────────────
                    "chunk_index":         chunk["chunk_index"],
                    "chunk_start_hms":     chunk["chunk_start_hms"],
                    "chunk_end_hms":       chunk["chunk_end_hms"],
                    "chunk_start_seconds": chunk["chunk_start_seconds"],
                    "chunk_end_seconds":   chunk["chunk_end_seconds"],
                    "segment_ids":         chunk["segment_ids"],
                    # ── content ───────────────────────────────────────
                    "text":                chunk["text"][:1000],
                    "doc_type":            "video_transcript",
                    "vehicle":             "Ford Mustang Mach-E 2026",
                    "namespace":           NAMESPACE,
                },
            })

        print(f"  Embedded batch {batch_start // EMBED_BATCH + 1} ({len(batch)} chunks)")

    # ── Upsert to Pinecone ────────────────────────────────────────────────
    for i in range(0, len(vectors), UPSERT_BATCH):
        batch_vecs = vectors[i : i + UPSERT_BATCH]
        index.upsert(vectors=batch_vecs, namespace=NAMESPACE)
        print(f"  Upserted vectors {i}–{i + len(batch_vecs) - 1}")

    total_upserted += len(vectors)
    print(f"  ✅ Done — {len(vectors)} vectors upserted from '{filename}'")

print(f"\n{'─'*60}")
print(f"Total vectors upserted this run: {total_upserted}")

## 9. Verify — Index Stats

In [ ]:
stats = index.describe_index_stats()
print("Index stats:")
print(f"  Total vectors : {stats['total_vector_count']}")
print("\nPer-namespace breakdown:")
for ns, ns_stats in stats.get("namespaces", {}).items():
    print(f"  {ns:35s} : {ns_stats['vector_count']} vectors")

## 10. Smoke Test — Similarity Search

In [ ]:
def query_transcript(question, k=3):
    q_embedding = embed_texts([question])[0]
    results = index.query(
        vector=q_embedding,
        top_k=k,
        namespace=NAMESPACE,
        include_metadata=True,
    )
    print(f"\n🔍  '{question}'")
    for match in results["matches"]:
        meta = match["metadata"]
        print(f"  [{match['score']:.4f}] "
              f"{meta['chunk_start_hms']}–{meta['chunk_end_hms']} "
              f"(segs {meta['segment_ids']}) "
              f"| video_id: {meta['video_id']} "
              f"| model: {meta['whisper_model']}")
        print(f"    tags: {meta.get('tags', [])}")
        print(f"    {meta['text'][:200]}...")

query_transcript("how to disconnect the high voltage system")
query_transcript("battery fire thermal runaway")

---
## Notes

| Parameter | Value | Rationale |
|---|---|---|
| `WINDOW_SIZE` | 10 segments | ~30–90 seconds of speech; covers a coherent topic block |
| `STEP_SIZE` | 8 | 2-segment overlap prevents context loss at chunk boundaries |
| `EMBED_BATCH` | 50 | Stays well within OpenAI's 2048-input limit per call |
| Namespace | `video_transcript_v2` | Enriched schema; `video_transcript` (old) left intact |
| Null fields | `""` / `0` | Pinecone rejects Python `None` — use empty string/zero |
| Metadata `text` | truncated to 1000 chars | Pinecone metadata cap is 40 KB per vector |

**To add a new video:**
1. Drop `*_segments.json` into `Talk/`
2. Add matching entry to `video_metadata.json` (use MP4 stem as key)
3. Re-run this notebook — upsert is idempotent, no duplicates created